In [ ]:
import demo

# "Beyond transcription" 

> Talk at AI Engineer London (April 2026)

## Sample phone conversation

In [ ]:
demo.ITranscription(
    audio=demo.AUDIO, 
    transcription=demo.GOLD_TRANSCRIPTION
)

## Speaker diarization

In [ ]:
# download pipeline from Huggingface
import os
from pyannote.audio import Pipeline
community1 = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-community-1', 
    token=os.environ.get('HF_TOKEN', None))

# use favorite pytorch backend (mps, cuda, cpu, ...)
import torch
_ = community1.to(torch.device('mps'))

# predict speaker diarization
prediction = community1(demo.AUDIO)

### Visualize what went wrong with `ipyannote`

In [ ]:
demo.IDiarizationErrorRate(
    audio=demo.AUDIO, 
    reference=demo.GOLD_DIARIZATION, 
    hypothesis=prediction.speaker_diarization, 
    permutation_invariant=True)

### Compute diarization error rate (DER) with `pyannote.metrics`

In [ ]:
from pyannote.metrics.diarization import DiarizationErrorRate
metric = DiarizationErrorRate()
der = metric(demo.GOLD_DIARIZATION, prediction.speaker_diarization, detailed=True)
demo.print(der)

### Improve diarization with `precision-2`

In [ ]:
# run precision2 diarization on pyannoteAI servers
precision2 = Pipeline.from_pretrained('pyannote/speaker-diarization-precision-2')
better_prediction = precision2(demo.AUDIO)

# compute diarization error rate
better_der = metric(demo.GOLD_DIARIZATION, better_prediction.speaker_diarization, detailed=True)
demo.print(better_der)

In [ ]:
demo.IDiarizationErrorRate(
    audio=demo.AUDIO, 
    reference=demo.GOLD_DIARIZATION, 
    hypothesis=better_prediction.speaker_diarization, 
    permutation_invariant=True)

## Speaker-attributed transcription

### Transcription (with Parakeet)

In [ ]:
parakeet = demo.parakeet(demo.AUDIO)

In [ ]:
demo.ITranscription(audio=demo.AUDIO, transcription=parakeet)

### Reconciliation based on timestamps (à la WhisperX)

In [ ]:
demo.Reconciliation(
    audio=demo.AUDIO, 
    transcript=parakeet, 
    diarization=better_prediction.speaker_diarization)

### Improve reconciliation with `STT orchestration`

In [ ]:
# instantiate pyannoteAI client
from pyannoteai.sdk import Client
client = Client()

# upload audio to pyannoteAI server
audio_url = client.upload(demo.AUDIO)

# create a job combining precision-2 diarization with parakeet STT
orchestration_job = client.diarize(
    audio_url, 
    model="precision-2",
    transcription=True,
    transcription_config={
        "model": "parakeet-tdt-0.6b-v3"
    })

# retrieve results
orchestration = client.retrieve(orchestration_job)
word_level_transcription = orchestration['output']['wordLevelTranscription']

In [ ]:
demo.ITranscription(audio=demo.AUDIO, transcription=word_level_transcription)